# Steps 1–3 — Data standardization and preprocessing (offset-preserving)

**Input:** `equal-data-set.xlsx` (a copy of the base workbook placed in this folder; columns *Job Advertisement* and *Source*).

**Outputs (all under `outputs/`):**
- `analysis/units.csv` — masked sentence-/requirement-level units, each linked to its advertisement, its component and its character span in the raw advertisement.
- `analysis/lines.csv` — every non-empty source line with its component, the rule that assigned it, and its raw character span (offsets only, no text).
- `analysis/ads.csv` — one row per advertisement (employer ID, parsing flags, unit counts); no raw text.
- `RESTRICTED_raw_and_contact/raw_layer.csv` — raw text, employer name, contact/application lines and source URL, kept apart from the analysis files.
- `validation_checks.csv`, `crosswalk_legacy_units.csv`, `figure_example_units.txt/.png`.

**What this notebook merges.** The earlier notebooks performed (Step 1) structural parsing, (Step 2) masking and sentence/requirement segmentation, and (Step 3) normalization. The parsing, masking and segmentation *rules* are reused unchanged. What changes is the bookkeeping: raw text is never edited; every unit stores `char_start`/`char_end` into the raw advertisement, so any later span label can be traced back to the original wording. Normalization is produced as a length-preserving view, so character positions in the normalized text still refer to the masked text and, through it, to the raw text.

## 1. Load the raw layer

The advertisement text is read exactly as supplied and fingerprinted with SHA-256. No cleaning is applied to it. The cell also profiles the characters that could break offset bookkeeping (carriage returns, the private-use character used as a segmentation placeholder, Unicode normalization form, and lowercase length changes).

In [1]:
from pathlib import Path
import hashlib
import json
import re
import unicodedata

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

DATA_PATH = Path("equal-data-set.xlsx")
OUT = Path("outputs")
ANALYSIS_DIR = OUT / "analysis"
RESTRICTED_DIR = OUT / "RESTRICTED_raw_and_contact"
LEGACY_UNITS = Path("..") / "outputs" / "equal_step2_masking" / "step2_masked_sentence_units.csv"
for d in (ANALYSIS_DIR, RESTRICTED_DIR):
    d.mkdir(parents=True, exist_ok=True)

raw_df = pd.read_excel(DATA_PATH, sheet_name=0, dtype=object)
raw_df.columns = [re.sub(r"\s+", " ", str(c)).strip() for c in raw_df.columns]
AD_COL, SRC_COL = raw_df.columns[0], raw_df.columns[1]

ads = pd.DataFrame(
    {
        "record_id": [f"row-{i + 2:04d}" for i in range(len(raw_df))],
        "raw_text": raw_df[AD_COL].map(lambda v: "" if pd.isna(v) else str(v)),
        "source_url": raw_df[SRC_COL].map(lambda v: "" if pd.isna(v) else str(v).strip()),
    }
)
ads["raw_sha256"] = ads["raw_text"].map(lambda t: hashlib.sha256(t.encode("utf-8")).hexdigest())

profile = pd.DataFrame(
    {
        "check": [
            "advertisements",
            "empty advertisement text",
            "missing source URL",
            "ads containing carriage return",
            "ads containing U+E000 (segmentation placeholder)",
            "ads changed by Unicode NFC normalization",
            "ads whose length changes under lower()",
            "ads with leading/trailing whitespace",
        ],
        "count": [
            len(ads),
            int((ads["raw_text"].str.strip() == "").sum()),
            int((ads["source_url"] == "").sum()),
            int(ads["raw_text"].str.contains("\r", regex=False).sum()),
            int(ads["raw_text"].str.contains("\ue000", regex=False).sum()),
            int((ads["raw_text"].map(lambda t: unicodedata.normalize("NFC", t)) != ads["raw_text"]).sum()),
            int((ads["raw_text"].map(lambda t: len(t.lower())) != ads["raw_text"].map(len)).sum()),
            int((ads["raw_text"] != ads["raw_text"].str.strip()).sum()),
        ],
    }
)
display(profile)
# Offset bookkeeping below assumes these hold; fail loudly if the data ever changes.
assert profile.loc[profile["check"].str.contains("carriage"), "count"].iloc[0] == 0
assert profile.loc[profile["check"].str.contains("U\\+E000"), "count"].iloc[0] == 0
assert profile.loc[profile["check"].str.contains("NFC"), "count"].iloc[0] == 0
assert profile.loc[profile["check"].str.contains("lower"), "count"].iloc[0] == 0

,check,count
0,advertisements,1500
1,empty advertisement text,0
2,missing source URL,1
3,ads containing carriage return,0
4,ads containing U+E000 (segmentation placeholder),0
5,ads changed by Unicode NFC normalization,0
6,ads whose length changes under lower(),0
7,ads with leading/trailing whitespace,19


## 2. Step 1 — Structural parsing with raw offsets

The rules are those of the earlier Step 1 (derived from the header and label lines observed in this workbook): the first non-empty line is the employer; the next line is the job title if it is short and is not a bullet, qualification header or contact line; a single top-to-bottom pass with one state (`body` → `qualification` → `contact`) assigns every remaining line to exactly one component. Cue matching is run on a whitespace-collapsed copy of each line (as before), but each line also keeps its **character span in the raw advertisement**.

Components: `employer_name`, `job_title`, `qualification`, `qualification_header`, `body_other`, `contact`.

In [2]:
QUAL_HEADER = re.compile(
    r"^(qu?a?l[a-z]*(?:\s+standards)?|requirements?|job\s+description)\s*:\s*(.*)$", re.I
)
CONTACT_HEADER_WORDS = re.compile(
    r"\b(send|resume|resumes|cv|apply|application|applications|applicants?|inquir\w*|"
    r"inquiries|contact|visit|information|e-?mail|mail|addressed|submit|office|address|"
    r"call|recruitment center|place assignment|file your|proceed)\b",
    re.I,
)
CONTACT_HEADER_PREFIX = re.compile(r"^requirements?\s+for\s+application\s*:", re.I)
EMAIL_RE = re.compile(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}")
URL_RE = re.compile(r"(?:https?://|www\.)\S+|\b[\w\-]+\.(?:com|ph|org|net|edu|gov)(?:/\S*)?\b", re.I)
PHONE_RE = re.compile(r"(?<!\d)(?:\+?63|0)?\d[\d\s\-()]{6,}\d(?!\d)")
LABEL_RE = re.compile(
    r"^(?:e-?mail(?:\s+ad+ress)?|email\s+adress|mobile\s+nos?\.?|telephone\s+nos?\.?|"
    r"telepne\s+nos?\.?|tel\.?\s*nos?\.?|contact\s+(?:nos?\.?|numbers?|person)|fax\s+nos?\.?|link|https)\s*:",
    re.I,
)
ADDRESS_CUE = re.compile(
    r"\b(street|st\.|avenue|ave\.?|road|rd\.?|highway|barangay|brgy\.?|purok|zone|"
    r"blvd\.?|boulevard|building|bldg\.?|floor|unit|city|province|municipality|poblacion|pob\.)\b",
    re.I,
)
BULLET_RE = re.compile(r"^•\s*")
APPLY_RE = re.compile(
    r"\b(?:interested|qualified)\b.{0,60}\b(?:applicants?|candidates?)\b"
    r"|\b(?:send|submit|email|forward|bring)\b.{0,40}\b(?:resumes?|cv|application|credentials|documents)\b"
    r"|\b(?:proceed|procedd|walk-?in)\b",
    re.I,
)


def is_contact_header(line):
    if CONTACT_HEADER_PREFIX.match(line):
        return True
    return line.endswith(":") and len(line) <= 110 and bool(CONTACT_HEADER_WORDS.search(line))


def has_contact_marker(line):
    return bool(LABEL_RE.match(line) or EMAIL_RE.search(line) or URL_RE.search(line) or PHONE_RE.search(line))


def std_line(line):
    # Cue-matching copy only: collapse blanks and map a leading list glyph to a bullet. Never stored as text.
    s = re.sub(r"[\t \u00a0]+", " ", line).strip()
    return re.sub(r"^[~●▪◦]\s*", "• ", s)


def clean_title(title):
    return re.sub(r"\s*\(\s*\d+\s*\)\s*$", "", title).strip()


def parse_ad(raw):
    lines, pos = [], 0
    for line_no, ln in enumerate(raw.split("\n"), start=1):
        start = pos + (len(ln) - len(ln.lstrip()))
        pos += len(ln) + 1
        stripped = ln.strip()
        if stripped:
            lines.append(
                {
                    "line_no": line_no,
                    "char_start": start,
                    "char_end": start + len(stripped),
                    "raw": stripped,
                    "std": std_line(ln),
                    "component": None,
                    "rule": None,
                    "seg_start": None,
                    "seg_end": None,
                }
            )
    flags = set()
    if not lines:
        return {"employer_name": "", "lines": lines, "flags": {"empty_text"}}

    def keep(L, component, rule, a=None, b=None):
        L["component"], L["rule"] = component, rule
        L["seg_start"] = L["char_start"] if a is None else a
        L["seg_end"] = L["char_end"] if b is None else b

    first = lines[0]
    first.update(component="employer_name", rule="FIRST_LINE")
    employer = re.sub(r"\s*•\s*$", "", first["std"]).strip()
    rest = lines[1:]

    title_found = False
    if rest:
        cand = rest[0]["std"]
        if (
            len(cand) <= 100
            and not BULLET_RE.match(cand)
            and not QUAL_HEADER.match(cand)
            and not is_contact_header(cand)
            and not has_contact_marker(cand)
        ):
            L = rest[0]
            keep(L, "job_title", "TITLE_LINE", b=L["char_start"] + len(clean_title(L["raw"])))
            rest = rest[1:]
            title_found = True
    if not title_found:
        flags.add("no_title")

    block, prev_bullet = "body", False
    has_qual = has_contact = False
    for L in rest:
        std = L["std"]
        m = QUAL_HEADER.match(std)
        if m:
            block = "qualification"
            if m.group(2).strip():
                mr = QUAL_HEADER.match(L["raw"])
                assert mr is not None
                keep(L, "qualification", "QUAL_HEADER", a=L["char_start"] + mr.start(2))
                has_qual = True
            else:
                L["component"], L["rule"] = "qualification_header", "QUAL_HEADER"
            prev_bullet = False
            continue
        if is_contact_header(std):
            block, has_contact = "contact", True
            L["component"], L["rule"] = "contact", "CONTACT_HEADER"
            prev_bullet = False
            continue
        if has_contact_marker(std):
            if block != "contact" and (len(std) > 100 or not LABEL_RE.match(std)) and not (
                EMAIL_RE.fullmatch(std) or URL_RE.fullmatch(std)
            ):
                flags.add("inline_contact_review")
            block, has_contact = "contact", True
            L["component"], L["rule"] = "contact", "CONTACT_MARKER"
            prev_bullet = False
            continue
        if not BULLET_RE.match(std) and APPLY_RE.search(std):
            block, has_contact = "contact", True
            flags.add("apply_instruction_review")
            L["component"], L["rule"] = "contact", "APPLY_INSTRUCTION"
            prev_bullet = False
            continue
        if block == "qualification" and prev_bullet and not BULLET_RE.match(std) and ADDRESS_CUE.search(std):
            block, has_contact = "contact", True
            flags.add("address_inferred")
            L["component"], L["rule"] = "contact", "ADDRESS_CUE"
            prev_bullet = False
            continue
        if block == "qualification":
            keep(L, "qualification", "IN_QUAL_BLOCK")
            has_qual = True
        elif block == "contact":
            L["component"], L["rule"] = "contact", "IN_CONTACT_BLOCK"
            has_contact = True
        else:
            keep(L, "body_other", "BODY")
        prev_bullet = bool(BULLET_RE.match(std))

    if not has_qual:
        flags.add("no_qualification_block")
    if not has_contact:
        flags.add("no_contact_block")
    return {"employer_name": employer, "lines": lines, "flags": flags}


parsed = ads["raw_text"].map(parse_ad)
ads["employer_name"] = parsed.map(lambda p: p["employer_name"])
ads["flags"] = parsed.map(lambda p: ";".join(sorted(p["flags"])))

line_rows = []
for rid, p in zip(ads["record_id"], parsed):
    for L in p["lines"]:
        line_rows.append(
            {
                "record_id": rid,
                "line_no": L["line_no"],
                "char_start": L["char_start"],
                "char_end": L["char_end"],
                "component": L["component"],
                "rule": L["rule"],
                "seg_start": L["seg_start"],
                "seg_end": L["seg_end"],
            }
        )
lines_df = pd.DataFrame(line_rows)
print(f"{len(lines_df):,} non-empty lines from {lines_df['record_id'].nunique():,} advertisements.")
display(lines_df["component"].value_counts(dropna=False).rename_axis("component").to_frame("lines"))
display(lines_df["rule"].value_counts(dropna=False).rename_axis("rule").to_frame("lines"))

14,819 non-empty lines from 1,500 advertisements.


,lines
component,
qualification,6281
contact,4098
employer_name,1500
job_title,1497
qualification_header,1347
body_other,96


,lines
rule,
IN_QUAL_BLOCK,6281
CONTACT_MARKER,2479
FIRST_LINE,1500
TITLE_LINE,1497
QUAL_HEADER,1347
ADDRESS_CUE,712
IN_CONTACT_BLOCK,516
CONTACT_HEADER,250
APPLY_INSTRUCTION,141


## 3. Step 2 — Masking and segmentation with an offset map

**Masking.** Employer names (full name and the name without a legal suffix such as *Inc.*/*Corp.*, each at least 4 characters) are replaced by `[EMPLOYER]`; e-mail addresses, URLs and phone-like strings found in retained text are replaced by `[EMAIL]`, `[URL]` and `[PHONE]`, in that order. Contact/application lines are not retained in the analysis text at all (they are represented by `[CONTACT]` at advertisement level and stored in the restricted layer).

Masking changes string lengths, so every masked character carries a pointer to the raw interval it came from. A placeholder points to the whole raw string it replaced. The raw intervals of masked units are therefore always exact, and each unit stores its placeholder replacements as `mask_edits` (`[raw_start, raw_end, placeholder]`).

**Segmentation.** Same rules as before: job titles stay whole; body and qualification lines are split at line/bullet boundaries and at sentence-ending punctuation followed by whitespace. Periods in decimals, dotted initials, common abbreviations and numbered-list markers are protected from splitting. Units without a letter are skipped. The leading list marker is excluded from the unit span.

In [3]:
BULLET_PREFIX_RE = re.compile(r"^\s*(?:[•●▪▫◦‣⁃○◉■□◆◇➢➤✓✔]+|[-*](?=\s|$))\s*")
STAGE1_BULLET_RE = re.compile(r"^[~●▪◦]\s*")
SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")
INITIALS_RE = re.compile(r"\b(?:[A-Za-z]\.){2,}")
ABBREVIATION_RE = re.compile(
    r"\b(?:Mr|Mrs|Ms|Dr|Prof|Sr|Jr|St|Ave|Blvd|No|Nos|Dept|Inc|Corp|Co|Ltd|etc|vs)\.(?=\s|$)", re.I
)
NUMBERED_LIST_RE = re.compile(r"^\s*\d+\.(?=\s)")
LETTER_RE = re.compile(r"[^\W\d_]")
HIDDEN = "\ue000"
COMPONENT_ORDER = ["job_title", "body_other", "qualification"]


def employer_variants(name):
    name = re.sub(r"\s+", " ", name).strip()
    if not name:
        return []
    core = re.sub(r"[,\s]+(?:inc|corp|corporation|incorporated|co|ltd|llc)\.?$", "", name, flags=re.I).strip(" ,.")
    return sorted((v for v in {name, core} if len(v) >= 4), key=len, reverse=True)


def employer_regex(variant):
    return re.compile(r"(?<!\w)" + re.sub(r"\\\s+|\\ ", r"\\s+", re.escape(variant)) + r"(?!\w)", re.I)


def employer_id(name):
    key = re.sub(r"\s+", " ", name).strip().lower()
    return "EMP-" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:8] if key else ""


def sub_with_map(text, cmap, regex, placeholder):
    # cmap[i] = (raw_start, raw_end, is_placeholder) for character i of text.
    out, omap, last = [], [], 0
    for m in regex.finditer(text):
        out.append(text[last : m.start()])
        omap += cmap[last : m.start()]
        span = cmap[m.start() : m.end()]
        rs, re_ = min(c[0] for c in span), max(c[1] for c in span)
        out.append(placeholder)
        omap += [(rs, re_, True)] * len(placeholder)
        last = m.end()
    out.append(text[last:])
    omap += cmap[last:]
    return "".join(out), omap


def mask_block(raw, seg_lines, employer):
    # Masking is applied to the whole component block (lines joined by newline), as in the earlier Step 2, so a
    # name or number that wraps across lines is still masked. Each character keeps its raw interval.
    text, cmap = "", []
    for k, L in enumerate(seg_lines):
        if k:
            text += "\n"
            cmap.append((seg_lines[k - 1]["seg_end"], L["seg_start"], False))
        seg = raw[L["seg_start"] : L["seg_end"]]
        text += seg
        cmap += [(L["seg_start"] + i, L["seg_start"] + i + 1, False) for i in range(len(seg))]
    for v in employer_variants(employer):
        text, cmap = sub_with_map(text, cmap, employer_regex(v), "[EMPLOYER]")
    for rx, ph in ((EMAIL_RE, "[EMAIL]"), (URL_RE, "[URL]"), (PHONE_RE, "[PHONE]")):
        text, cmap = sub_with_map(text, cmap, rx, ph)
    return text, cmap

def strip_leading_marker(s):
    # Reproduces the earlier two-stage behaviour: stage 1 turns one leading ~ ● ▪ ◦ into a bullet, stage 2 strips bullets.
    # Returns the index in s where the unit text starts.
    m1 = STAGE1_BULLET_RE.match(s)
    if m1:
        m2 = BULLET_PREFIX_RE.match("•" + s[m1.end() :])
        return m1.end() + m2.end() - 1
    m2 = BULLET_PREFIX_RE.match(s)
    return m2.end() if m2 else 0

def hide_periods(text):
    text = re.sub(r"(?<=\d)\.(?=\d)", HIDDEN, text)
    text = INITIALS_RE.sub(lambda m: m.group(0).replace(".", HIDDEN), text)
    text = ABBREVIATION_RE.sub(lambda m: m.group(0).replace(".", HIDDEN), text)
    return NUMBERED_LIST_RE.sub(lambda m: m.group(0).replace(".", HIDDEN), text)


def piece_spans(text, whole):
    # Spans (start, end) in text of the units. Length-preserving, so spans index the masked string directly.
    def trim(a, b, s):
        seg = s[a:b]
        return a + len(seg) - len(seg.lstrip()), a + len(seg.rstrip())

    if whole:
        a, b = trim(0, len(text), text)
        return [(a, b)] if b > a else []
    h = hide_periods(text)
    bounds, prev = [], 0
    for m in SENTENCE_SPLIT_RE.finditer(h):
        bounds.append((prev, m.start()))
        prev = m.end()
    bounds.append((prev, len(h)))
    out = []
    for a, b in bounds:
        a, b = trim(a, b, h)
        if b > a and LETTER_RE.search(h[a:b]):
            out.append((a, b))
    return out


def edits_for(cmap, text, p0, p1):
    edits, i = [], p0
    while i < p1:
        if cmap[i][2]:
            j = i
            while j < p1 and cmap[j][2] and cmap[j][:2] == cmap[i][:2]:
                j += 1
            edits.append([cmap[i][0], cmap[i][1], text[i:j]])
            i = j
        else:
            i += 1
    return edits


def build_units(raw, parsed_ad, rid, emp_id):
    rows, idx = [], 0
    for comp in COMPONENT_ORDER:
        seg_lines = [L for L in parsed_ad["lines"] if L["component"] == comp]
        if not seg_lines:
            continue
        starts = [L["seg_start"] for L in seg_lines]
        text, cmap = mask_block(raw, seg_lines, parsed_ad["employer_name"])
        pos = 0
        for mline in text.split("\n"):
            line_start = pos
            pos += len(mline) + 1
            if not mline.strip():
                continue
            a = strip_leading_marker(mline)
            sub = mline[a:]
            for p0, p1 in piece_spans(sub, whole=(comp == "job_title")):
                g0, g1 = line_start + a + p0, line_start + a + p1
                idx += 1
                tm = text[g0:g1]
                tn = re.sub(r"\s", " ", tm).lower()
                assert len(tn) == len(tm)
                k = max(i for i, s in enumerate(starts) if s <= cmap[g0][0])
                rows.append(
                    {
                        "unit_id": f"{rid}-s{idx:03d}",
                        "record_id": rid,
                        "employer_id": emp_id,
                        "unit_index": idx,
                        "component": comp,
                        "line_no": seg_lines[k]["line_no"],
                        "char_start": cmap[g0][0],
                        "char_end": cmap[g1 - 1][1],
                        "text_masked": tm,
                        "text_norm": tn,
                        "mask_edits": json.dumps(edits_for(cmap, text, g0, g1), ensure_ascii=False),
                    }
                )
    return rows

### Behaviour checks on synthetic inputs

These use made-up strings (not data from the workbook) to confirm the segmentation rules, the marker handling and the offset map before they are applied to the real advertisements.

In [4]:
def seg_texts(line, whole=False):
    return [line[a:b] for a, b in piece_spans(line, whole)]


assert seg_texts("B.S. Psychology. Must be 18 years old.") == ["B.S. Psychology.", "Must be 18 years old."]
assert seg_texts("1.5 years experience!") == ["1.5 years experience!"]
assert seg_texts("Sr. Sales Manager", whole=True) == ["Sr. Sales Manager"]
assert seg_texts("Female/Male; with 2 years' experience") == ["Female/Male; with 2 years' experience"]

demo = "ACME Corp\nCashier (3)\nQualifications:\n~ Female\n~ Age 18-25.  ACME Corp offers training.\nSend resume to hr@acme.com"
dp = parse_ad(demo)
assert dp["employer_name"] == "ACME Corp"
assert [L["component"] for L in dp["lines"]] == [
    "employer_name", "job_title", "qualification_header", "qualification", "qualification", "contact"
]
du = build_units(demo, dp, "demo", "EMP-demo")
assert [u["text_masked"] for u in du] == ["Cashier", "Female", "Age 18-25.", "[EMPLOYER] offers training."]
assert demo[du[0]["char_start"] : du[0]["char_end"]] == "Cashier"
assert demo[du[1]["char_start"] : du[1]["char_end"]] == "Female"
assert demo[du[3]["char_start"] : du[3]["char_end"]] == "ACME Corp offers training."
assert json.loads(du[3]["mask_edits"]) == [[demo.index("ACME Corp offers"), demo.index("ACME Corp offers") + len("ACME Corp"), "[EMPLOYER]"]]
print("Synthetic behaviour checks passed.")

Synthetic behaviour checks passed.


In [5]:
unit_rows = []
for rid, raw, p in zip(ads["record_id"], ads["raw_text"], parsed):
    unit_rows += build_units(raw, p, rid, employer_id(p["employer_name"]))
units = pd.DataFrame(unit_rows)
ads["employer_id"] = ads["employer_name"].map(employer_id)
ads["n_units"] = ads["record_id"].map(units["record_id"].value_counts()).fillna(0).astype(int)
print(f"{len(units):,} units from {units['record_id'].nunique():,} advertisements.")
display(units["component"].value_counts().rename_axis("component").to_frame("units"))
display(units.head(8)[["unit_id", "component", "line_no", "char_start", "char_end", "text_masked", "text_norm"]])

7,971 units from 1,500 advertisements.


,units
component,
qualification,6374
job_title,1497
body_other,100


,unit_id,component,line_no,char_start,char_end,text_masked,text_norm
0,row-0002-s001,job_title,3,8,29,Production Associates,production associates
1,row-0002-s002,qualification,6,53,145,At least High School Graduate or 2-year Vocational Graduate or 2-year ~College Undergraduate,at least high school graduate or 2-year vocational graduate or 2-year ~college undergraduate
2,row-0002-s003,qualification,7,147,178,With or without work experience,with or without work experience
3,row-0002-s004,qualification,8,180,214,Must be willing to render overtime,must be willing to render overtime
4,row-0003-s001,job_title,3,10,34,Human Resource Assistant,human resource assistant
5,row-0003-s002,qualification,6,53,64,Female/Male,female/male
6,row-0003-s003,qualification,7,66,172,"Graduate of BS Psychology, BSBA Human Resource Management, BS ~Behavioral Science or other related courses","graduate of bs psychology, bsba human resource management, bs ~behavioral science or other related courses"
7,row-0003-s004,qualification,8,175,235,With at least 1 year experience in human resources functions,with at least 1 year experience in human resources functions


## 4. Step 3 — Normalization as a length-preserving view

Each unit's `text_norm` is its masked text with every whitespace character replaced by a single space and letters lowercased; no characters are added, removed or reordered, no words are filtered, and punctuation is kept. Because `len(text_norm) == len(text_masked)` for every unit, a character span found in `text_norm` is the same span in `text_masked`, and `mask_edits` carries it back to the raw advertisement. The line-leading list marker is excluded from the unit span itself (see Step 2). Original case remains available in `text_masked` for case-sensitive models.

In [6]:
def masked_span_to_raw(row, a, b):
    # Map a character span [a, b) of text_masked/text_norm to a raw-advertisement span [start, end).
    edits = json.loads(row["mask_edits"])
    pos_m, pos_r = 0, row["char_start"]
    result = {}
    for rs, re_, ph in edits:
        gap = rs - pos_r
        for key, x in (("a", a), ("b", b)):
            if key not in result and x <= pos_m + gap:
                result[key] = pos_r + (x - pos_m)
        if "a" not in result and a < pos_m + gap + len(ph):
            result["a"] = rs
        if "b" not in result and b <= pos_m + gap + len(ph):
            result["b"] = re_
        pos_m += gap + len(ph)
        pos_r = re_
    for key, x in (("a", a), ("b", b)):
        if key not in result:
            result[key] = pos_r + (x - pos_m)
    return result["a"], result["b"]


def reconstruct_masked(raw, row):
    s, e = row["char_start"], row["char_end"]
    text = raw[s:e]
    for rs, re_, ph in sorted(json.loads(row["mask_edits"]), reverse=True):
        text = text[: rs - s] + ph + text[re_ - s :]
    return text

## 5. Validation

Automatic checks that the traceability claims hold on every unit: (1) the raw layer is unchanged; (2) every non-empty line has exactly one component; (3) each unit's raw span, with its placeholder replacements applied, reproduces the unit's masked text exactly; (4) units do not overlap and keep document order within an advertisement; (5) normalization preserves length; (6) no e-mail, URL or phone-like string and no employer name remains in unit text; (7) the analysis files contain no raw-text column; (8) mapping a span back to the raw text works for every placeholder.

In [7]:
raw_by_id = dict(zip(ads["record_id"], ads["raw_text"]))
emp_by_id = dict(zip(ads["record_id"], ads["employer_name"]))
checks = []


def record(name, ok, detail=""):
    checks.append({"check": name, "passed": bool(ok), "detail": detail})


# 1 raw layer unchanged vs a fresh read
fresh = pd.read_excel(DATA_PATH, sheet_name=0, dtype=object)[AD_COL].map(lambda v: "" if pd.isna(v) else str(v))
record("raw text identical to a fresh read of the workbook", (fresh.values == ads["raw_text"].values).all())

# 2 line accounting
n_nonempty = int(ads["raw_text"].map(lambda t: sum(1 for l in t.split("\n") if l.strip())).sum())
record(
    "every non-empty line has exactly one component",
    len(lines_df) == n_nonempty and lines_df["component"].notna().all(),
    f"{len(lines_df)} lines table rows vs {n_nonempty} non-empty raw lines",
)

# 3 raw slice + edits reproduces masked text
bad = [r.unit_id for r in units.itertuples() if reconstruct_masked(raw_by_id[r.record_id], r._asdict()) != r.text_masked]
record("raw span + mask_edits reproduces text_masked for every unit", not bad, f"{len(bad)} mismatches")
bad_plain = [
    r.unit_id
    for r in units.itertuples()
    if r.mask_edits == "[]" and raw_by_id[r.record_id][r.char_start : r.char_end] != r.text_masked
]
record("units without placeholders equal raw[char_start:char_end]", not bad_plain, f"{len(bad_plain)} mismatches")

# 4 order and overlap
viol = 0
for rid, g in units.groupby("record_id"):
    spans = list(zip(g["char_start"], g["char_end"]))
    viol += sum(1 for (s1, e1), (s2, e2) in zip(spans, spans[1:]) if s2 < e1)
    viol += sum(1 for s, e in spans if s >= e)
record("units are non-empty, ordered and non-overlapping within each advertisement", viol == 0, f"{viol} violations")
record("unit_id is unique", units["unit_id"].is_unique)

# 5 normalization
record("text_norm has the same length as text_masked", (units["text_norm"].str.len() == units["text_masked"].str.len()).all())
record("text_norm is lowercase with no tab/newline", units["text_norm"].map(lambda t: t == t.lower() and not re.search(r"[\t\n]", t)).all())

# 6 leakage
contact_leak = units["text_masked"].map(lambda t: bool(EMAIL_RE.search(t) or URL_RE.search(t) or PHONE_RE.search(t)))
emp_leak = [
    any(re.search(r"(?<!\w)" + re.escape(v) + r"(?!\w)", t, re.I) for v in employer_variants(emp_by_id[rid]))
    for rid, t in zip(units["record_id"], units["text_masked"])
]
record("no e-mail/URL/phone-like string in unit text", int(contact_leak.sum()) == 0, f"{int(contact_leak.sum())} units")
record("no employer name in unit text", sum(emp_leak) == 0, f"{sum(emp_leak)} units")

# 7 analysis files carry no raw text
record(
    "analysis tables have no raw-text, employer-name, contact or source columns",
    not ({"raw_text", "employer_name", "contact_text", "source_url"} & (set(units.columns) | set(lines_df.columns))),
)

# 8 span mapping through placeholders
n_edit_units, map_bad = 0, 0
for r in units[units["mask_edits"] != "[]"].itertuples():
    n_edit_units += 1
    row = r._asdict()
    search_from = 0
    for rs, re_, ph in json.loads(r.mask_edits):
        i = r.text_masked.find(ph, search_from)
        search_from = i + len(ph)
        a, b = masked_span_to_raw(row, i, i + len(ph))
        if (a, b) != (rs, re_):
            map_bad += 1
    # an unmasked word outside any placeholder maps to identical raw text
    m = re.search(r"[A-Za-z]{3,}", r.text_masked.replace("[EMPLOYER]", "").replace("[EMAIL]", "").replace("[URL]", "").replace("[PHONE]", ""))
    if m:
        w = m.group(0)
        i = r.text_masked.find(w)
        a, b = masked_span_to_raw(row, i, i + len(w))
        if raw_by_id[r.record_id][a:b] != w:
            map_bad += 1
record("span mapping round trip for units containing placeholders", map_bad == 0, f"{n_edit_units} units tested, {map_bad} failures")

validation = pd.DataFrame(checks)
display(validation)
assert validation["passed"].all(), "A validation check failed - see table above."

,check,passed,detail
0,raw text identical to a fresh read of the workbook,True,
1,every non-empty line has exactly one component,True,14819 lines table rows vs 14819 non-empty raw lines
2,raw span + mask_edits reproduces text_masked for every unit,True,0 mismatches
3,units without placeholders equal raw[char_start:char_end],True,0 mismatches
4,"units are non-empty, ordered and non-overlapping within each advertisement",True,0 violations
5,unit_id is unique,True,
6,text_norm has the same length as text_masked,True,
7,text_norm is lowercase with no tab/newline,True,
8,no e-mail/URL/phone-like string in unit text,True,0 units
9,no employer name in unit text,True,0 units


### Masking summary

In [8]:
summary = pd.DataFrame(
    {
        "measure": [
            "ads with [EMPLOYER] in retained units",
            "ads with [EMAIL]/[URL]/[PHONE] in retained units",
            "units containing a placeholder",
            "ads with employer name recorded (restricted layer)",
            "ads with contact/application lines (excluded from analysis text)",
            "ads with a source URL (restricted layer)",
            "ads whose first line begins with a list glyph (employer string keeps it)",
        ],
        "count": [
            int(units[units["text_masked"].str.contains("[EMPLOYER]", regex=False)]["record_id"].nunique()),
            int(units[units["text_masked"].str.contains(r"\[(?:EMAIL|URL|PHONE)\]")]["record_id"].nunique()),
            int((units["mask_edits"] != "[]").sum()),
            int((ads["employer_name"] != "").sum()),
            int(lines_df[lines_df["component"] == "contact"]["record_id"].nunique()),
            int((ads["source_url"] != "").sum()),
            int(ads["employer_name"].str.startswith("•").sum()),
        ],
    }
)
display(summary)
display(ads["flags"].str.get_dummies(sep=";").sum().rename_axis("flag").to_frame("ads"))

,measure,count
0,ads with [EMPLOYER] in retained units,16
1,ads with [EMAIL]/[URL]/[PHONE] in retained units,0
2,units containing a placeholder,16
3,ads with employer name recorded (restricted layer),1500
4,ads with contact/application lines (excluded from analysis text),1218
5,ads with a source URL (restricted layer),1499
6,ads whose first line begins with a list glyph (employer string keeps it),0


,ads
flag,
address_inferred,712
apply_instruction_review,138
inline_contact_review,170
no_contact_block,282
no_qualification_block,157
no_title,3


## 6. Crosswalk to the earlier sentence-unit file

The earlier Steps 1–2 produced `step2_masked_sentence_units.csv`, which downstream notebooks use. Units here are numbered in the same order (job title, body, qualification), so a crosswalk by ID is attempted and checked by text. Whitespace is compared in collapsed form because the earlier file collapsed internal blanks. Differences are listed, not hidden.

In [9]:
if LEGACY_UNITS.is_file():
    old = pd.read_csv(LEGACY_UNITS, encoding="utf-8-sig", dtype=str, keep_default_na=False)
    compact = lambda s: re.sub(r"\s+", " ", s).strip()
    new = units[["unit_id", "record_id", "component", "text_masked"]].copy()
    new["new_compact"] = new["text_masked"].map(compact)
    old = old.rename(columns={"sentence_id": "unit_id", "component": "old_component", "sentence": "old_text"})
    xw = new.merge(old[["unit_id", "old_component", "old_text"]], on="unit_id", how="outer", indicator=True)
    xw["status"] = "text_match"
    xw.loc[xw["_merge"] == "left_only", "status"] = "only_in_new"
    xw.loc[xw["_merge"] == "right_only", "status"] = "only_in_legacy"
    both = xw["_merge"] == "both"
    xw.loc[both & ((xw["new_compact"] != xw["old_text"]) | (xw["component"] != xw["old_component"])), "status"] = "differs"
    print(f"Legacy units: {len(old):,}; new units: {len(new):,}")
    display(xw["status"].value_counts().rename_axis("status").to_frame("units"))
    diff_recs = xw.loc[xw["status"] != "text_match", "unit_id"].str.rsplit("-", n=1).str[0].nunique()
    print(f"Advertisements with at least one non-matching unit: {diff_recs:,}")
    display(xw[xw["status"] != "text_match"].head(25)[["unit_id", "status", "component", "old_component", "new_compact", "old_text"]])
else:
    xw = pd.DataFrame()
    print(f"Legacy unit file not found at {LEGACY_UNITS}; crosswalk skipped.")

Legacy units: 7,971; new units: 7,971


,units
status,
text_match,7971


Advertisements with at least one non-matching unit: 0


,unit_id,status,component,old_component,new_compact,old_text


## 7. Example of sentence-/requirement-level data

One real advertisement from the dataset whose job title is *Cashier*, shown in the `[component]  text` form used in the manuscript. The selection is deterministic: the first advertisement with a qualification unit equal to *Female* is preferred, otherwise the first *Cashier* advertisement.

In [10]:
cashier = units[(units["component"] == "job_title") & (units["text_masked"].str.strip().str.lower() == "cashier")]
print(f"Advertisements with job title 'Cashier': {cashier['record_id'].nunique()}")
pref = [
    rid
    for rid in cashier["record_id"]
    if ((units["record_id"] == rid) & (units["component"] == "qualification") & (units["text_norm"] == "female")).any()
]
print(f"...of which contain a qualification unit equal to 'female': {len(pref)}")
EXAMPLE_RECORD_ID = (pref or list(cashier["record_id"]))[0]
ex = units[units["record_id"] == EXAMPLE_RECORD_ID]
fig_lines = [f"[{r.component}]  {r.text_masked}" for r in ex.itertuples()]
print(f"\n{EXAMPLE_RECORD_ID}  ->  {len(ex)} units\n")
print("\n".join(fig_lines))
display(ex[["unit_id", "component", "line_no", "char_start", "char_end", "text_masked"]])

Advertisements with job title 'Cashier': 12
...of which contain a qualification unit equal to 'female': 6

row-0038  ->  5 units

[job_title]  Cashier
[qualification]  Female
[qualification]  Graduate of any 4-year course or at leasr 2 years in college
[qualification]  With pleasing personality
[qualification]  With or without experience


,unit_id,component,line_no,char_start,char_end,text_masked
193,row-0038-s001,job_title,3,10,17,Cashier
194,row-0038-s002,qualification,6,40,46,Female
195,row-0038-s003,qualification,7,48,108,Graduate of any 4-year course or at leasr 2 years in college
196,row-0038-s004,qualification,8,110,135,With pleasing personality
197,row-0038-s005,qualification,9,137,163,With or without experience


## 8. Export

In [11]:
units.to_csv(ANALYSIS_DIR / "units.csv", index=False, encoding="utf-8-sig")
lines_df.to_csv(ANALYSIS_DIR / "lines.csv", index=False, encoding="utf-8-sig")
ads[["record_id", "employer_id", "flags", "n_units", "raw_sha256"]].to_csv(
    ANALYSIS_DIR / "ads.csv", index=False, encoding="utf-8-sig"
)

contact_by_id = {
    rid: "\n".join(L["raw"] for L in p["lines"] if L["component"] == "contact") for rid, p in zip(ads["record_id"], parsed)
}
restricted = ads[["record_id", "employer_id", "employer_name", "source_url", "raw_text", "raw_sha256"]].copy()
restricted["contact_text"] = restricted["record_id"].map(contact_by_id)
restricted.to_csv(RESTRICTED_DIR / "raw_layer.csv", index=False, encoding="utf-8-sig")

validation.to_csv(OUT / "validation_checks.csv", index=False, encoding="utf-8-sig")
if len(xw):
    xw.drop(columns=["_merge"]).to_csv(OUT / "crosswalk_legacy_units.csv", index=False, encoding="utf-8-sig")
(OUT / "figure_example_units.txt").write_text("\n".join(fig_lines) + "\n", encoding="utf-8")

try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(9, 0.38 * len(fig_lines) + 0.6))
    ax.axis("off")
    ax.text(0, 1, "\n".join(fig_lines), va="top", ha="left", family="monospace", fontsize=10, transform=ax.transAxes)
    fig.savefig(OUT / "figure_example_units.png", dpi=200, bbox_inches="tight")
    plt.close(fig)
except Exception as exc:
    print("Figure PNG not written:", exc)

config = {
    "input": str(DATA_PATH),
    "input_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "n_ads": len(ads),
    "n_lines": len(lines_df),
    "n_units": len(units),
    "offsets": "char_start/char_end index the raw advertisement text (Python str, 0-based, end-exclusive)",
    "normalization": "whitespace -> single space per character, lowercase; length-preserving",
}
(OUT / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Saved:", *sorted(str(p) for p in OUT.rglob("*") if p.is_file()), sep="\n  ")

Saved:
  outputs\RESTRICTED_raw_and_contact\raw_layer.csv
  outputs\analysis\ads.csv
  outputs\analysis\lines.csv
  outputs\analysis\units.csv
  outputs\config.json
  outputs\crosswalk_legacy_units.csv
  outputs\figure_example_units.png
  outputs\figure_example_units.txt
  outputs\validation_checks.csv
